# Group 3: regression 

## Setup: load local data and regression helpers


In [2]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from risk_model_betas import regress, tsat_beta
from stepwise_market_sector_betas import (
    read_table, read_companies, aligned_series, passes_tsat,
    standard_deviation, SECTOR_TO_ETF, REMAINING_SECTOR_T_THRESHOLD,
)

DATA = ROOT / "data"
STAGGER = "stagger_3"
EXAMPLE = "VIST"
companies = read_companies(DATA / "companies.csv")
return_table = read_table(DATA / f"{STAGGER}_returns.csv")
factor_table = read_table(DATA / f"{STAGGER}_sector_factors.csv")
dates = sorted(return_table["date"])
returns = {c: aligned_series(return_table, dates, c) for c in return_table if c != "date"}
sectors = {c: aligned_series(factor_table, dates, c) for c in factor_table if c != "date"}
stocks = list(companies)

pd.set_option("display.max_rows", 8)
pd.set_option("display.max_columns", 8)
pd.set_option("display.float_format", lambda value: f"{value:.4f}")

def fit_stage(x, y):
    fit = regress(x, y)
    pairs = [(a, b) for a, b in zip(x, y) if a is not None and b is not None]
    fit["alpha"] = sum(b - fit["beta"] * a for a, b in pairs) / len(pairs)
    return fit

def fit_row(ticker, factor, fit):
    return {"ticker": ticker, "factor": factor, "beta_raw": fit["beta"],
            "tstat": fit["tstat"], "nobs": fit["nobs"]}

def show_residuals(before, after):
    display(pd.DataFrame({"date": dates, "input": before[EXAMPLE],
                          "residual": after[EXAMPLE]}).head(4))

print(f"{STAGGER}: {len(stocks)} stocks, {len(dates)} dates; example: {EXAMPLE}.")

stagger_3: 43 stocks, 80 dates; example: VIST.


## 1. Regress foreign stocks on their local currency


In [3]:
currency_fits = {}
after_currency = {t: returns[t][:] for t in stocks}
currency_rows = []
for ticker in stocks:
    currency = companies[ticker]["currency"]
    if currency == "USD":
        continue
    factor = f"{currency}/USD"
    fit = fit_stage(returns[factor], returns[ticker])
    currency_fits[ticker] = fit
    kept = passes_tsat(fit)
    if kept:
        after_currency[ticker] = fit["residuals"][:]
    currency_rows.append({**fit_row(ticker, factor, fit),
                          "beta_kept": tsat_beta(fit), "kept": kept})

currency_betas = pd.DataFrame(currency_rows).set_index("ticker")
display(currency_betas[["beta_raw", "tstat", "beta_kept"]].head(8))
print(f"{len(currency_fits)} foreign stocks fitted; {len(stocks) - len(currency_fits)} US stocks skip this stage.")
show_residuals(returns, after_currency)

,beta_raw,tstat,beta_kept
ticker,,,
VIST,0.5446,0.9980,0.5446
STX,3.7775,2.9504,0.0000
NTR,-0.2535,-1.5754,-0.2535
ASND,0.5775,1.0247,0.5775
NOK,0.9622,1.3241,0.9622


5 foreign stocks fitted; 38 US stocks skip this stage.


,date,input,residual
0,2020-08-03,0.0455,0.0105
1,2020-08-31,-0.1796,-0.2250
2,2020-09-28,-0.0203,-0.0535
3,2020-10-26,-0.1133,-0.1514


## 2. Regress the carried-forward returns on AUD


In [4]:
aud_fits = {}
aud_rows = []
for ticker in stocks:
    fit = fit_stage(returns["AUD/USD"], after_currency[ticker])
    aud_fits[ticker] = fit
    aud_rows.append(fit_row(ticker, "AUD/USD", fit))

aud_raw_betas = pd.DataFrame(aud_rows).set_index("ticker")
display(aud_raw_betas[["beta_raw", "tstat"]].head(8))
display(pd.DataFrame({"date": dates,
                      "input_to_AUD": after_currency[EXAMPLE],
                      "AUD_fitted_residual_before_filter": aud_fits[EXAMPLE]["residuals"]}).head(5))

,beta_raw,tstat
ticker,,
VST,1.0530,1.8064
VIST,0.5782,0.9686
AEP,0.1334,0.5390
ETR,0.2121,0.8128
DIS,1.3025,3.4081
CMG,0.9721,2.0171
AMZN,0.7993,1.9858
FIX,1.3233,2.3648


,date,input_to_AUD,AUD_fitted_residual_before_filter
0,2020-08-03,0.0105,0.0021
1,2020-08-31,-0.2250,-0.2290
2,2020-09-28,-0.0535,-0.0497
3,2020-10-26,-0.1514,-0.1441
4,2020-11-23,0.1214,0.1071


## 3. Apply the AUD t-statistic rule


In [5]:
after_aud = {}
aud_filter_rows = []
for ticker in stocks:
    fit = aud_fits[ticker]
    kept = passes_tsat(fit)
    after_aud[ticker] = (fit["residuals"] if kept else after_currency[ticker])[:]
    aud_filter_rows.append({
        "ticker": ticker, "beta_raw": fit["beta"], "tstat": fit["tstat"],
        "beta_kept": tsat_beta(fit), "kept": kept,
        "next_stage_input": "AUD residual" if kept else "pre-AUD returns",
    })

aud_betas = pd.DataFrame(aud_filter_rows).set_index("ticker")
display(aud_betas[["beta_raw", "tstat", "beta_kept"]].head(8))
print(f"AUD fits retained: {int(aud_betas['kept'].sum())} of {len(stocks)}.")
show_residuals(after_currency, after_aud)

,beta_raw,tstat,beta_kept
ticker,,,
VST,1.0530,1.8064,1.0530
VIST,0.5782,0.9686,0.5782
AEP,0.1334,0.5390,0.1334
ETR,0.2121,0.8128,0.2121
DIS,1.3025,3.4081,0.0000
CMG,0.9721,2.0171,0.0000
AMZN,0.7993,1.9858,0.7993
FIX,1.3233,2.3648,0.0000


AUD fits retained: 23 of 43.


,date,input,residual
0,2020-08-03,0.0105,0.0021
1,2020-08-31,-0.2250,-0.2290
2,2020-09-28,-0.0535,-0.0497
3,2020-10-26,-0.1514,-0.1441


## 4. Regress the residuals on SPY and keep beta


In [6]:
spy_fits = {}
after_spy = {}
spy_rows = []
for ticker in stocks:
    fit = fit_stage(returns["SPY"], after_aud[ticker])
    spy_fits[ticker] = fit
    after_spy[ticker] = fit["residuals"][:]
    spy_rows.append({**fit_row(ticker, "SPY", fit), "beta_kept": fit["beta"]})

spy_betas = pd.DataFrame(spy_rows).set_index("ticker")
display(spy_betas[["beta_raw", "tstat", "beta_kept"]].head(8))
show_residuals(after_aud, after_spy)

,beta_raw,tstat,beta_kept
ticker,,,
VST,1.1139,3.2971,1.1139
VIST,0.2585,0.7021,0.2585
AEP,-0.1263,-0.8285,-0.1263
ETR,0.1745,1.0888,0.1745
DIS,1.5531,8.5120,1.5531
CMG,1.2932,4.8168,1.2932
AMZN,1.1961,5.7244,1.1961
FIX,1.3772,4.2680,1.3772


,date,input,residual
0,2020-08-03,0.0021,-0.0020
1,2020-08-31,-0.2290,-0.2291
2,2020-09-28,-0.0497,-0.0433
3,2020-10-26,-0.1441,-0.1377


## 5. Regress on the stock's prior (own) sector factor


In [7]:
own_sector_fits = {}
own_factors = {}
after_own_sector = {}
own_sector_rows = []
for ticker in stocks:
    factor = SECTOR_TO_ETF[companies[ticker]["sector"]] + "_residual"
    own_factors[ticker] = factor
    fit = fit_stage(sectors[factor], after_spy[ticker])
    own_sector_fits[ticker] = fit
    after_own_sector[ticker] = fit["residuals"][:]
    own_sector_rows.append({
        **fit_row(ticker, factor, fit), "beta_kept": fit["beta"],
        "residual_risk": standard_deviation(fit["residuals"]),
    })

own_sector_betas = pd.DataFrame(own_sector_rows).set_index("ticker")
display(own_sector_betas[["beta_raw", "tstat", "beta_kept"]].head(8))
show_residuals(after_spy, after_own_sector)

,beta_raw,tstat,beta_kept
ticker,,,
VST,1.1486,3.7497,1.1486
VIST,0.9953,6.0360,0.9953
AEP,1.0997,13.0933,1.0997
ETR,1.0963,11.2457,1.0963
DIS,0.9105,3.7139,0.9105
CMG,1.1301,3.3228,1.1301
AMZN,1.0661,4.1687,1.0661
FIX,1.5490,4.1367,1.5490


,date,input,residual
0,2020-08-03,-0.0020,-0.0035
1,2020-08-31,-0.2291,-0.1765
2,2020-09-28,-0.0433,0.0527
3,2020-10-26,-0.1377,-0.1178


## 6. Stepwise regression on the remaining sectors


In [8]:
remaining_fits = {}
remaining_unselected = {}
final_residuals = {}
selection_rows = []
trial_rows = []
path_rows = []

for ticker in stocks:
    unused = sorted(f for f in sectors if f != own_factors[ticker])
    current = [
        value if value is not None and all(sectors[f][i] is not None for f in unused)
        else None
        for i, value in enumerate(after_own_sector[ticker])
    ]
    selected = {}
    unselected = {}
    step = 1
    while unused:
        candidates = []
        unselected = {}
        iteration_rows = []
        for factor in unused:
            fit = regress(sectors[factor], current)
            unselected[factor] = fit
            tstat = fit["tstat"]
            qualifies = abs(tstat) > REMAINING_SECTOR_T_THRESHOLD
            rss = sum(v ** 2 for v in fit["residuals"] if v is not None)
            iteration_rows.append({
                "ticker": ticker, "step": step, "factor": factor,
                "beta_raw": fit["beta"],
                "tstat": tstat, "RSS": rss, "qualifies": qualifies,
            })
            if qualifies:
                candidates.append((rss, factor, fit))

        winner = min(candidates, key=lambda item: (item[0], item[1])) if candidates else None
        for row in iteration_rows:
            row["selected_this_step"] = winner is not None and row["factor"] == winner[1]
        trial_rows.extend(iteration_rows)
        if winner is None:
            break

        _, factor, fit = winner
        selected[factor] = fit
        current = fit["residuals"][:]
        selection_rows.append({
            "ticker": ticker, "step": step, "factor": factor,
            "beta_kept": fit["beta"], "tstat_at_entry": fit["tstat"],
            "nobs": fit["nobs"], "residual_risk_after_step": standard_deviation(current),
        })
        unused.remove(factor)
        del unselected[factor]
        step += 1

    remaining_fits[ticker] = selected
    remaining_unselected[ticker] = unselected
    final_residuals[ticker] = current
    path_rows.append({
        "ticker": ticker, "selected_count": len(selected),
        "selection_order": ";".join(selected),
        "risk_before": standard_deviation(after_own_sector[ticker]),
        "risk_after": standard_deviation(current),
    })

selection_paths = pd.DataFrame(path_rows).set_index("ticker")
selected_betas = pd.DataFrame(selection_rows, columns=[
    "ticker", "step", "factor", "beta_kept", "tstat_at_entry", "nobs", "residual_risk_after_step",
])
stepwise_trials = pd.DataFrame(trial_rows)
display(selection_paths.head(8))
display(selected_betas.head(8))
print(f"First remaining-sector trials for {EXAMPLE}:")
display(stepwise_trials.loc[stepwise_trials["ticker"] == EXAMPLE].head(8))
show_residuals(after_own_sector, final_residuals)

,selected_count,selection_order,risk_before,risk_after
ticker,,,,
VST,3,XLV_residual;XLRE_residual;XLI_residual,0.1090,0.0943
VIST,2,XLB_residual;XLY_residual,0.1065,0.0970
AEP,1,XLV_residual,0.0299,0.0282
ETR,0,,0.0347,0.0347
DIS,1,XLF_residual,0.0590,0.0566
CMG,0,,0.0881,0.0881
AMZN,1,XLB_residual,0.0662,0.0583
FIX,4,XLV_residual;XLC_residual;XLU_residual;XLF_res...,0.1024,0.0771


,ticker,step,factor,beta_kept,tstat_at_entry,nobs,residual_risk_after_step
0,VST,1,XLV_residual,-1.0278,-3.3842,80,0.1018
1,VST,2,XLRE_residual,-0.7786,-2.3453,80,0.0984
2,VST,3,XLI_residual,0.9028,2.6181,80,0.0943
3,VIST,1,XLB_residual,-0.9196,-3.3786,80,0.0995
4,VIST,2,XLY_residual,-0.7581,-2.0244,80,0.0970
5,AEP,1,XLV_residual,0.2614,3.1081,80,0.0282
6,DIS,1,XLF_residual,0.5011,2.5551,80,0.0566
7,AMZN,1,XLB_residual,-0.7602,-4.7687,80,0.0583


First remaining-sector trials for VIST:


,ticker,step,factor,beta_raw,tstat,RSS,qualifies,selected_this_step
34,VIST,1,XLB_residual,-0.9196,-3.3786,0.7822,True,True
35,VIST,1,XLC_residual,-0.6630,-1.5184,0.8709,False,False
36,VIST,1,XLF_residual,-0.2507,-0.6814,0.8913,False,False
37,VIST,1,XLI_residual,-0.6266,-1.6361,0.8669,False,False
38,VIST,1,XLK_residual,0.5061,1.2272,0.8797,False,False
39,VIST,1,XLP_residual,-0.6382,-1.6508,0.8664,False,False
40,VIST,1,XLRE_residual,-0.3376,-0.9445,0.8865,False,False
41,VIST,1,XLU_residual,-0.1485,-0.4968,0.8938,False,False


,date,input,residual
0,2020-08-03,-0.0035,0.0094
1,2020-08-31,-0.1765,-0.1490
2,2020-09-28,0.0527,0.0775
3,2020-10-26,-0.1178,-0.1065


## Final beta table


In [9]:
final_rows = []
for ticker in stocks:
    row = {
        "ticker": ticker, "stagger": STAGGER,
        "currency_beta": tsat_beta(currency_fits.get(ticker)),
        "aud_beta": tsat_beta(aud_fits[ticker]),
        "spy_beta": spy_fits[ticker]["beta"],
        "sector_factor": own_factors[ticker],
        "sector_beta": own_sector_fits[ticker]["beta"],
        "nobs": spy_fits[ticker]["nobs"],
        "remaining_sector_order": ";".join(remaining_fits[ticker]),
        "remaining_sector_count": len(remaining_fits[ticker]),
        "remaining_nobs": sum(v is not None for v in final_residuals[ticker]),
        "residual_risk_before_remaining": standard_deviation(after_own_sector[ticker]),
        "residual_risk": standard_deviation(final_residuals[ticker]),
    }
    for factor in sorted(sectors):
        fit = remaining_fits[ticker].get(factor, remaining_unselected[ticker].get(factor))
        prefix = "remaining_" + factor.removesuffix("_residual")
        row[prefix + "_beta"] = (
            None if fit is None or fit["tstat"] is None
            else fit["beta"] if factor in remaining_fits[ticker] else 0.0
        )
        row[prefix + "_tstat"] = None if fit is None else fit["tstat"]
    final_rows.append(row)

final_betas = pd.DataFrame(final_rows).set_index("ticker")
display(final_betas[["currency_beta", "aud_beta", "spy_beta", "sector_beta",
                     "remaining_sector_count", "residual_risk"]].head(8))
print("Additional sector betas (zero = tested but unselected):")
display(selected_betas[["ticker", "factor", "beta_kept", "tstat_at_entry"]].head(8))

print(f"Selected {len(selected_betas)} additional sector betas across {len(stocks)} stocks.")

,currency_beta,aud_beta,spy_beta,sector_beta,remaining_sector_count,residual_risk
ticker,,,,,,
VST,NaN,1.0530,1.1139,1.1486,3,0.0943
VIST,0.5446,0.5782,0.2585,0.9953,2,0.0970
AEP,NaN,0.1334,-0.1263,1.0997,1,0.0282
ETR,NaN,0.2121,0.1745,1.0963,0,0.0347
DIS,NaN,0.0000,1.5531,0.9105,1,0.0566
CMG,NaN,0.0000,1.2932,1.1301,0,0.0881
AMZN,NaN,0.7993,1.1961,1.0661,1,0.0583
FIX,NaN,0.0000,1.3772,1.5490,4,0.0771


Additional sector betas (zero = tested but unselected):


,ticker,factor,beta_kept,tstat_at_entry
0,VST,XLV_residual,-1.0278,-3.3842
1,VST,XLRE_residual,-0.7786,-2.3453
2,VST,XLI_residual,0.9028,2.6181
3,VIST,XLB_residual,-0.9196,-3.3786
4,VIST,XLY_residual,-0.7581,-2.0244
5,AEP,XLV_residual,0.2614,3.1081
6,DIS,XLF_residual,0.5011,2.5551
7,AMZN,XLB_residual,-0.7602,-4.7687


Selected 64 additional sector betas across 43 stocks.
